# Séptima sesión mejorada — Portales públicos peruanos, Selenium y automatización

**Curso:** Fundamentos de Programación para IA Generativa Aplicada  
**Programa:** Especialización en IA Generativa aplicada a CCSS y Gestión Pública  

## Propósito de la sesión

Esta sesión organiza y mejora el notebook `S6_P2_Webscraping_peruvian_portals`, tratándolo como un bloque aplicado de scraping gubernamental.

Aquí el objetivo ya no es simplemente aprender a scrapear una página, sino:

- navegar portales públicos peruanos,
- identificar estructuras dinámicas,
- recuperar tablas y documentos,
- manipular URLs de consulta,
- construir funciones reutilizables,
- y exportar resultados de forma ordenada y reproducible.

## Idea metodológica

Esta sesión muestra una transición clave:

**scraping exploratorio → scraping estructurado → scraping parametrizado**

Es decir, se pasa de localizar elementos visibles a comprender cómo funciona internamente la lógica de navegación del portal.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import pandas as pd
import numpy as np
import os
import time
import json
import shutil

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 200)

RUTA_BASE = Path("/content/drive/MyDrive/IA Generativa para CCSS/Base de datos")
RUTA_RESULTADOS = RUTA_BASE / "Resultados" / "Sesion_7_Portales_Peruanos"

RUTA_TABLAS = RUTA_RESULTADOS / "tablas"
RUTA_AUDITORIA = RUTA_RESULTADOS / "auditoria"
RUTA_PDFS = RUTA_RESULTADOS / "pdfs"
RUTA_LOGS = RUTA_RESULTADOS / "logs"

for ruta in [RUTA_RESULTADOS, RUTA_TABLAS, RUTA_AUDITORIA, RUTA_PDFS, RUTA_LOGS]:
    ruta.mkdir(parents=True, exist_ok=True)

print("Resultados en:", RUTA_RESULTADOS)

Mounted at /content/drive
Resultados en: /content/drive/MyDrive/IA Generativa para CCSS/Base de datos/Resultados/Sesion_7_Portales_Peruanos


In [2]:
!pip -q install selenium webdriver-manager openpyxl pandas numpy lxml unidecode
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!dpkg -i google-chrome-stable_current_amd64.deb
!apt-get install -f -y

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.3/510.3 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 8.1 MB/s eta 0:00:00
Selecting previously unselected package google-chrome-stable.
(Reading database ... 118194 files and directories currently installed.)
Preparing to unpack google-chrome-stable_current_amd64.deb ...
Unpacking google-chrome-stable (147.0.7727.55-1) ...
dpkg: dependency problems prevent configuration of google-chrome-stable:
 google-chrome-stable depends on libatk-bridge2.0-0 (>= 2.5.3); however:
  Package libatk-bridge2.0-0 is not installed.
 google-chrome-stable depends on libatk1.0-0 (>= 2.11.90); however:
  Package libatk1.0-0 is not installed.
 google-chrome-stable depends on libatspi2.0-0 (>= 2.9.90); however:
  Package libatspi2.0-0 is not installed.
 google-chrome-sta

In [3]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

def crear_driver(headless=True):
    opciones = Options()
    if headless:
        opciones.add_argument("--headless")
    opciones.add_argument("--no-sandbox")
    opciones.add_argument("--disable-dev-shm-usage")
    opciones.add_argument("--disable-gpu")
    opciones.add_argument("--window-size=1920,1080")
    opciones.add_argument("--remote-debugging-port=9222")
    opciones.binary_location = "/usr/bin/google-chrome"

    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=opciones)
    driver.maximize_window()
    return driver

driver = crear_driver()
print("Driver listo.")

Driver listo.


In [5]:
from selenium.webdriver.common.by import By
from io import StringIO
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

url_congreso = "https://wb2server.congreso.gob.pe/spley-portal/#/expediente/search"
driver.get(url_congreso)

# Add an explicit wait for the table element to be present
wait = WebDriverWait(driver, 20) # Wait up to 20 seconds
table_element = wait.until(EC.presence_of_element_located((By.XPATH, '/html/body/app-root/app-publico/div[3]/app-search/section/div/div/p-table/div/div/table')))

table_html = table_element.get_attribute("outerHTML")
table_df = pd.read_html(StringIO(table_html))[0]

display(table_df.head())
table_df.to_excel(RUTA_TABLAS / "s7_congreso_tabla_principal.xlsx", index=False)

,PROYECTOS DE LEY,FECHA DE PRESENTACIÓN,TÍTULO,ESTADO PROCESAL,PROPONENTE,AUTORES
0,Proyecto de Ley 14376/2025-CR,Fecha de Presentación07/04/2026,TítuloPROYECTO DE LEY QUE REQUIERE CERTIFICADO DE ORIENTACIÓN PRENUPCIAL COMO REQUISITO PARA CONTRAER MATRIMONIO CIVIL,Estado ProcesalPRESENTADO,ProponenteCongreso,"AutoresLaura Rojas, Judith Arriola Tueros, José Alberto Burgos Oliveros, Juan Bartolomé ver más..."
1,Proyecto de Ley 14375/2025-CR,Fecha de Presentación07/04/2026,TítuloPROYECTO DE LEY QUE DECLARA DE INTERÉS NACIONAL Y DE NECESIDAD PÚBLICA EL FORTALECIMIENTO E IMPLEMENTACIÓN DE LA UNIVERSIDAD NACIONAL TECNOLÓGICA DE SAN JUAN DE LURIGANCHO,Estado ProcesalPRESENTADO,ProponenteCongreso,"AutoresLaura Rojas, Judith Arriola Tueros, José Alberto Bellido Ugarte, Guido ver más..."
2,Proyecto de Ley 14374/2025-CR,Fecha de Presentación07/04/2026,TítuloPROYECTO DE LEY QUE CREA LA UNIVERSIDAD NACIONAL INTERCULTURAL TECNOLÓGICA DE ESPINAR,Estado ProcesalPRESENTADO,ProponenteCongreso,"AutoresMontalvo Cubas, Segundo Toribio Cruz Mamani, Flavio Espiritu Cavero, Raul ver más..."
3,Proyecto de Ley 14373/2025-CR,Fecha de Presentación07/04/2026,TítuloPROYECTO DE LEY QUE AUTORIZA EL RETIRO EXTRAORDINARIO Y FACULTATIVO DE HASTA CUATRO (4) UIT DE LOS APORTES AL SISTEMA NACIONAL DE PENSIONES,Estado ProcesalPRESENTADO,ProponenteCongreso,"AutoresMontalvo Cubas, Segundo Toribio Cruz Mamani, Flavio Espiritu Cavero, Raul ver más..."
4,Proyecto de Ley 14372/2025-CR,Fecha de Presentación07/04/2026,"TítuloPROYECTO DE LEY QUE DECLARA DE INTERÉS NACIONAL Y NECESIDAD PÚBLICA EL MEJORAMIENTO DEL CAMINO VECINAL CHOSGÓN, TRAMO CARRETERA FERNANDO BELAÚNDE TERRY - LOCALIDAD DE CHOSGÓN, DISTRITO DE JA...",Estado ProcesalPRESENTADO,ProponenteCongreso,"AutoresMontalvo Cubas, Segundo Toribio Cruz Mamani, Flavio Espiritu Cavero, Raul ver más..."


In [6]:
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

wait = WebDriverWait(driver, 20)

tablas_paginas = []

for index in range(1, 6):
    table_element = wait.until(
        EC.presence_of_element_located((By.XPATH, '/html/body/app-root/app-publico/div[3]/app-search/section/div/div/p-table/div/div/table'))
    )
    table_html = table_element.get_attribute("outerHTML")
    tabla_temp = pd.read_html(StringIO(table_html))[0]
    tabla_temp["pagina_scrapeada"] = index
    tablas_paginas.append(tabla_temp)

    next_button = driver.find_element(By.XPATH, '/html/body/app-root/app-publico/div[3]/app-search/section/div/div/p-table/div/p-paginator[1]/div/button[3]')
    next_button.click()
    time.sleep(2)

df_congreso_paginas = pd.concat(tablas_paginas, ignore_index=True)
display(df_congreso_paginas.head())
df_congreso_paginas.to_excel(RUTA_TABLAS / "s7_congreso_paginas_1_5.xlsx", index=False)

,PROYECTOS DE LEY,FECHA DE PRESENTACIÓN,TÍTULO,ESTADO PROCESAL,PROPONENTE,AUTORES,pagina_scrapeada
0,Proyecto de Ley 14376/2025-CR,Fecha de Presentación07/04/2026,TítuloPROYECTO DE LEY QUE REQUIERE CERTIFICADO DE ORIENTACIÓN PRENUPCIAL COMO REQUISITO PARA CONTRAER MATRIMONIO CIVIL,Estado ProcesalPRESENTADO,ProponenteCongreso,"AutoresLaura Rojas, Judith Arriola Tueros, José Alberto Burgos Oliveros, Juan Bartolomé ver más...",1
1,Proyecto de Ley 14375/2025-CR,Fecha de Presentación07/04/2026,TítuloPROYECTO DE LEY QUE DECLARA DE INTERÉS NACIONAL Y DE NECESIDAD PÚBLICA EL FORTALECIMIENTO E IMPLEMENTACIÓN DE LA UNIVERSIDAD NACIONAL TECNOLÓGICA DE SAN JUAN DE LURIGANCHO,Estado ProcesalPRESENTADO,ProponenteCongreso,"AutoresLaura Rojas, Judith Arriola Tueros, José Alberto Bellido Ugarte, Guido ver más...",1
2,Proyecto de Ley 14374/2025-CR,Fecha de Presentación07/04/2026,TítuloPROYECTO DE LEY QUE CREA LA UNIVERSIDAD NACIONAL INTERCULTURAL TECNOLÓGICA DE ESPINAR,Estado ProcesalPRESENTADO,ProponenteCongreso,"AutoresMontalvo Cubas, Segundo Toribio Cruz Mamani, Flavio Espiritu Cavero, Raul ver más...",1
3,Proyecto de Ley 14373/2025-CR,Fecha de Presentación07/04/2026,TítuloPROYECTO DE LEY QUE AUTORIZA EL RETIRO EXTRAORDINARIO Y FACULTATIVO DE HASTA CUATRO (4) UIT DE LOS APORTES AL SISTEMA NACIONAL DE PENSIONES,Estado ProcesalPRESENTADO,ProponenteCongreso,"AutoresMontalvo Cubas, Segundo Toribio Cruz Mamani, Flavio Espiritu Cavero, Raul ver más...",1
4,Proyecto de Ley 14372/2025-CR,Fecha de Presentación07/04/2026,"TítuloPROYECTO DE LEY QUE DECLARA DE INTERÉS NACIONAL Y NECESIDAD PÚBLICA EL MEJORAMIENTO DEL CAMINO VECINAL CHOSGÓN, TRAMO CARRETERA FERNANDO BELAÚNDE TERRY - LOCALIDAD DE CHOSGÓN, DISTRITO DE JA...",Estado ProcesalPRESENTADO,ProponenteCongreso,"AutoresMontalvo Cubas, Segundo Toribio Cruz Mamani, Flavio Espiritu Cavero, Raul ver más...",1


In [7]:
pagination_next_button = driver.find_element(By.XPATH, '/html/body/app-root/app-publico/div[3]/app-search/section/div/div/p-table/div/p-paginator[1]/div/button[4]')
pagination_next_button.click()
time.sleep(2)

last_pagination_button = driver.find_element(By.XPATH, '/html/body/app-root/app-publico/div[3]/app-search/section/div/div/p-table/div/p-paginator[1]/div/span[2]/button[5]')
n_pagination_buttons = last_pagination_button.text

pd.DataFrame([{"n_pagination_buttons": n_pagination_buttons}]).to_csv(
    RUTA_AUDITORIA / "s7_congreso_paginacion.csv", index=False
)

print("Número de botones de paginación:", n_pagination_buttons)

Número de botones de paginación: 288


In [8]:
bills_id = table_df["PROYECTOS DE LEY"].apply(lambda x: x.split("Ley ")[1].split("/")[0]).tolist()
base_url = "https://wb2server.congreso.gob.pe/spley-portal/#/expediente/2021/"

descargas = []

for bill in bills_id[:8]:
    bill_url = base_url + bill
    driver.get(bill_url)
    time.sleep(5)

    try:
        pdf_button = driver.find_element(By.XPATH, '//*[@id="p-tabpanel-0"]/p-table/div/div/table/tbody/tr/td[5]/button')
        pdf_button.click()
        descargas.append({"bill": bill, "status": "ok", "url": bill_url})
    except Exception as e:
        descargas.append({"bill": bill, "status": "error", "url": bill_url, "error": str(e)})

    time.sleep(5)

df_descargas_congreso = pd.DataFrame(descargas)
display(df_descargas_congreso)
df_descargas_congreso.to_csv(RUTA_AUDITORIA / "s7_congreso_descarga_pdfs.csv", index=False, encoding="utf-8")

,bill,status,url
0,14376,ok,https://wb2server.congreso.gob.pe/spley-portal/#/expediente/2021/14376
1,14375,ok,https://wb2server.congreso.gob.pe/spley-portal/#/expediente/2021/14375
2,14374,ok,https://wb2server.congreso.gob.pe/spley-portal/#/expediente/2021/14374
3,14373,ok,https://wb2server.congreso.gob.pe/spley-portal/#/expediente/2021/14373
4,14372,ok,https://wb2server.congreso.gob.pe/spley-portal/#/expediente/2021/14372
5,14371,ok,https://wb2server.congreso.gob.pe/spley-portal/#/expediente/2021/14371
6,14370,ok,https://wb2server.congreso.gob.pe/spley-portal/#/expediente/2021/14370
7,14369,ok,https://wb2server.congreso.gob.pe/spley-portal/#/expediente/2021/14369


In [9]:
driver.quit()
driver = crear_driver()

from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

url_siaf = "https://apps5.mineco.gob.pe/transparencia/Navegador/default.aspx?y=2023&ap=ActProy"
driver.get(url_siaf)
wait = WebDriverWait(driver, 20)

frames = driver.find_elements(By.TAG_NAME, "frame")
if frames:
    driver.switch_to.frame("frame0")

wait.until(EC.element_to_be_clickable((By.ID, "ctl00_CPH1_BtnTipoGobierno"))).click()
wait.until(EC.element_to_be_clickable((By.XPATH, "//*[@id='ctl00_CPH1_RptData_ctl03_TD0']/input"))).click()

wait.until(EC.element_to_be_clickable((By.ID, "ctl00_CPH1_BtnSector"))).click()
wait.until(EC.element_to_be_clickable((By.XPATH, "//*[@id='ctl00_CPH1_RptData_ctl02_TD0']/input"))).click()

wait.until(EC.element_to_be_clickable((By.ID, "ctl00_CPH1_BtnPliego"))).click()

table = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "table.Data")))
html = table.get_attribute("outerHTML")
df = pd.read_html(StringIO(html))[0]

headers_elements_r0 = driver.find_element(By.XPATH, '//*[@id="ctl00_CPH1_Mt0_Row0"]').find_elements(By.TAG_NAME, 'td')
headers_r0 = [element.text for element in headers_elements_r0]

headers_elements_r1 = driver.find_element(By.XPATH, '//*[@id="ctl00_CPH1_Mt0_Row1"]').find_elements(By.TAG_NAME, 'td')
headers_r1 = [element.text for element in headers_elements_r1]

new_columns = []
new_columns.append(headers_r0[0].strip() if headers_r0[0].strip() else 'Radio_Button')

for i in range(1, 6):
    new_columns.append(headers_r0[i].strip())

ejecucion_base = headers_r0[6].strip()
for sub_header in headers_r1:
    new_columns.append(f"{ejecucion_base}_{sub_header.strip()}")

new_columns.append(headers_r0[7].strip())
new_columns = [col.replace(" ", "_").replace("%", "Porcentaje").replace(".", "") for col in new_columns]

df.columns = new_columns
display(df.head())
df.to_excel(RUTA_TABLAS / "s7_siaf_regional.xlsx", index=False)

,Radio_Button,Pliego,PIA,PIM,Certificación,Compromiso_Anual,Ejecución_Atención_de_Compromiso_Mensual,Ejecución_Devengado,Ejecución_Girado,Avance_Porcentaje
0,NaN,440: GOBIERNO REGIONAL DEL DEPARTAMENTO DE AMAZONAS,1179543064,1540608616,1532841226,1506168202,1501812047,1435345927,1434621250,93.2
1,NaN,441: GOBIERNO REGIONAL DEL DEPARTAMENTO DE ANCASH,2535471411,3034507325,2700692982,2574885651,2573348946,2573085975,2568206679,84.8
2,NaN,442: GOBIERNO REGIONAL DEL DEPARTAMENTO DE APURIMAC,1219427445,1481066763,1450345046,1439674071,1427867460,1424312522,1424042430,96.2
3,NaN,443: GOBIERNO REGIONAL DEL DEPARTAMENTO DE AREQUIPA,2380402279,3199066208,3048096940,2988856508,2957285909,2956159533,2955071813,92.4
4,NaN,444: GOBIERNO REGIONAL DEL DEPARTAMENTO DE AYACUCHO,1739949908,2245901927,2188831480,2177688126,2166802546,2163798948,2163545202,96.3


In [10]:
def siaf_scraper(year, spending_type, activity_type, path):
    from selenium.webdriver.support.ui import WebDriverWait
    from selenium.webdriver.support import expected_conditions as EC
    from selenium.webdriver.common.by import By

    activity_mappings = {
        "Actividades": "Actividad",
        "Proyectos": "Proyecto",
        "Actividades y proyectos": "ActProy"
    }

    activity_type_code = activity_mappings.get(activity_type)

    url = f"https://apps5.mineco.gob.pe/transparencia/Navegador/Navegar_7.aspx?0=&1=M&37=M&7=&y={year}&ap={activity_type_code}&cpage=1&psize=2000"

    driver = crear_driver()
    driver.get(url)
    wait = WebDriverWait(driver, 10)

    xpath_mappings = {
        "Categoría Presupuestal": '//*[@id="ctl00_CPH1_BtnProgramaPpto"]',
        "Producto o Proyecto": '//*[@id="ctl00_CPH1_BtnProdProy"]',
        "Función": '//*[@id="ctl00_CPH1_BtnFuncion"]'
    }

    xpath = xpath_mappings.get(spending_type)

    os.makedirs(path, exist_ok=True)
    path_txt = os.path.join(path, f"{year}_{spending_type}_records.txt")

    with open(path_txt, "w", encoding="utf-8") as f:
        n_iterations = driver.find_elements(By.XPATH, "//tr[contains(@id, 'tr')]")

        for index, district in enumerate(n_iterations):
            try:
                entities_list = wait.until(
                    EC.presence_of_all_elements_located((By.XPATH, "//tr[contains(@id, 'tr')]"))
                )
                entity_button = entities_list[index]
                full_entity_name = entity_button.find_element(By.XPATH, './td[2]').text.strip()
                short_entity_name = full_entity_name.split(":")[1].strip() if ":" in full_entity_name else full_entity_name

                entity_button.click()
                spending_type_button = wait.until(EC.element_to_be_clickable((By.XPATH, xpath)))
                spending_type_button.click()

                headers_elements_r0 = wait.until(
                    EC.presence_of_element_located((By.XPATH, '//*[@id="ctl00_CPH1_Mt0_Row0"]'))
                ).find_elements(By.TAG_NAME, 'td')
                headers_elements_r1 = wait.until(
                    EC.presence_of_element_located((By.XPATH, '//*[@id="ctl00_CPH1_Mt0_Row1"]'))
                ).find_elements(By.TAG_NAME, 'td')

                headers_r0 = [e.text for e in headers_elements_r0 if e.text not in ['Ejecución', 'Avance % ']]
                headers_r1 = [e.text for e in headers_elements_r1]
                headers = headers_r0 + headers_r1 + ['Avance %', 'Municipalidad', 'year', 'tipo_actividad']

                table_element = wait.until(EC.element_to_be_clickable((By.XPATH, "//table[@class='Data']")))
                table_html = table_element.get_attribute("outerHTML")
                df = pd.read_html(StringIO(table_html))[0]

                df["Municipalidad"] = full_entity_name
                df["year"] = year
                df["tipo_actividad"] = activity_type_code
                df.columns = headers
                df = df.drop(df.columns[0], axis=1)
                df = df[df.columns[-3:].tolist() + df.columns[:-3].tolist()]

                folder_path = os.path.join(path, f"{year}_{spending_type}")
                os.makedirs(folder_path, exist_ok=True)
                file_path = os.path.join(folder_path, f"{year}_{short_entity_name}.xlsx")
                df.to_excel(file_path, index=False)

                print(f"Extraído: {short_entity_name}")
                f.write(f"Extraído: {short_entity_name}\n")

                come_back_button = wait.until(
                    EC.element_to_be_clickable((By.XPATH, '//*[@id="ctl00_CPH1_RptHistory_ctl04_TD0"]'))
                )
                come_back_button.click()

            except Exception as e:
                print(f"Error en índice {index}: {e}")
                f.write(f"Error en índice {index}: {e}\n")
                continue

    driver.quit()

In [ ]:
path_siaf_complete = str(RUTA_TABLAS / "siaf_complete")

year = "2021"
spending_type = "Categoría Presupuestal"
activity_type = "Actividades y proyectos"

# Úsalo con cuidado: puede tardar bastante.
siaf_scraper(year, spending_type, activity_type, path_siaf_complete)

Extraído: MUNICIPALIDAD PROVINCIAL DE CHACHAPOYAS
Extraído: MUNICIPALIDAD DISTRITAL DE ASUNCION
Extraído: MUNICIPALIDAD DISTRITAL DE BALSAS
Extraído: MUNICIPALIDAD DISTRITAL DE CHETO
Extraído: MUNICIPALIDAD DISTRITAL DE CHILIQUIN
Extraído: MUNICIPALIDAD DISTRITAL DE CHUQUIBAMBA
Extraído: MUNICIPALIDAD DISTRITAL DE GRANADA
Extraído: MUNICIPALIDAD DISTRITAL DE HUANCAS
Extraído: MUNICIPALIDAD DISTRITAL DE LA JALCA
Error en índice 9: Message: stale element reference: stale element not found
  (Session info: chrome=147.0.7727.55); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#staleelementreferenceexception
Stacktrace:
#0 0x573fc47e1b6a <unknown>
#1 0x573fc41e3265 <unknown>
#2 0x573fc41f729b <unknown>
#3 0x573fc41f5d31 <unknown>
#4 0x573fc41eaae6 <unknown>
#5 0x573fc41e8ca9 <unknown>
#6 0x573fc41ec95b <unknown>
#7 0x573fc41eca03 <unknown>
#8 0x573fc4235959 <unknown>
#9 0x573fc42361b1 <unknown>
#10 0x573fc422af26 <unknow